# 3. Parallel & Conditional Chains
**Industry:** Insurance

Analyze a claim's damage and fraud risk in parallel, then route to auto-approve or manual review.

In [1]:
!pip install langchain langchain-openai python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
import os
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = AzureChatOpenAI(azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o"), api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"))

damage_prompt = ChatPromptTemplate.from_template("Assess the damage severity (Low/Medium/High) for: {claim}")
fraud_prompt = ChatPromptTemplate.from_template("Assess fraud risk (Score 1-10) for: {claim}. Only return the number.")

parallel_chain = RunnableParallel(
    damage=damage_prompt | llm | StrOutputParser(),
    fraud_score=fraud_prompt | llm | StrOutputParser()
)

def route_claim(data):
    try:
        score = int(data['fraud_score'].strip())
    except:
        score = 10
    if score < 4 and 'High' not in data['damage']:
        return "Auto-Approve: Claim seems legitimate and damage is not severe."
    else:
        return "Manual Review Required: High fraud risk or severe damage detected."

full_chain = parallel_chain | RunnableLambda(route_claim)

claim_desc = "Car was rear-ended at a stop light. Bumper is dented."
print(full_chain.invoke({"claim": claim_desc}))

Auto-Approve: Claim seems legitimate and damage is not severe.
